# 第二部分：数据清洗

本 Notebook 完成以下清洗步骤：
1. **单表清洗**：缺失值检测与处理、日期格式统一、数据类型检查、重复值处理、离群值标注
2. **宽表与长表转换**
3. **多表合并**（个股 + 指数 + 宏观）
4. **数据存储**（CSV + Parquet）

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

print(f"数据清洗开始时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

stocks = [
    {"code": "000001", "name": "平安银行", "industry": "银行"},
    {"code": "600036", "name": "招商银行", "industry": "银行"},
    {"code": "002594", "name": "比亚迪",   "industry": "汽车"},
    {"code": "300750", "name": "宁德时代", "industry": "能源"},
    {"code": "601012", "name": "隆基绿能", "industry": "能源"},
    {"code": "600519", "name": "贵州茅台", "industry": "白酒"},
    {"code": "000063", "name": "中兴通讯", "industry": "通讯"},
    {"code": "002352", "name": "顺丰控股", "industry": "物流"},
    {"code": "600048", "name": "保利发展", "industry": "房地产"},
    {"code": "002475", "name": "立讯精密", "industry": "通讯"},
]
stocks_df = pd.DataFrame(stocks)

## 3.1 单表清洗

对每只股票的原始数据依次执行 6 个清洗步骤，**每步均展示清洗前后的变化**。

### 步骤 1：缺失值检测

统计每列缺失值的数量和比例，分析缺失的可能原因。

In [ ]:
# 读取样例数据，展示缺失值检测过程
sample = pd.read_csv("data/stock/stock_000001.csv")
print("清洗前 - 平安银行数据：")
print(f"  行数: {len(sample)}, 列数: {len(sample.columns)}")
print(f"  列名: {sample.columns.tolist()}")

missing = sample.isnull().sum()
missing_pct = (missing / len(sample) * 100).round(2)
missing_df = pd.DataFrame({"缺失数量": missing, "缺失比例(%)": missing_pct})
has_missing = missing.sum() > 0
if has_missing:
    display(missing_df[missing_df["缺失数量"] > 0])
else:
    print("无缺失值")
print("\n可能原因：股票数据缺失通常因停牌、节假日或数据源更新延迟。")

### 步骤 2-6：完整清洗流程

| 步骤 | 操作 | 说明 |
|------|------|------|
| 2 | 缺失值处理 | 前向填充（停牌期间价格不变，前值合理） |
| 3 | 日期格式 | 统一为 datetime64 并设为索引 |
| 4 | 数据类型 | 确保价格、成交量列为 float64 |
| 5 | 重复值 | 删除重复日期行（保留最后一条） |
| 6 | 离群值 | 对日对数收益率超过 ln(1.20) 的标注 is_extreme=True，不删除 |

In [ ]:
def clean_stock(filepath, code, name):
    """对单只股票执行完整的 6 步清洗"""
    df = pd.read_csv(filepath)
    report = {"code": code, "name": name, "原始行数": len(df)}

    # 步骤 2：缺失值处理 - 前向填充
    price_cols = ["open", "close", "high", "low", "volume", "turnover"]
    for col in price_cols:
        if col in df.columns:
            before = df[col].isnull().sum()
            df[col] = df[col].ffill()
            after = df[col].isnull().sum()
            if before > 0:
                print(f"    {name} {col}: 缺失 {before} -> 填充后 {after}")

    # 步骤 3：日期格式统一为 datetime64
    date_col = "date" if "date" in df.columns else "日期"
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.rename(columns={date_col: "date"})
    df = df.set_index("date")

    # 步骤 4：数据类型检查
    for col in price_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # 步骤 5：重复值处理
    dups = df.index.duplicated().sum()
    report["重复值数"] = dups
    df = df[~df.index.duplicated(keep="last")]
    report["去重后行数"] = len(df)

    # 步骤 6：离群值标注（日涨跌幅超过 +-20%）
    df["log_return"] = np.log(df["close"] / df["close"].shift(1))
    df["is_extreme"] = df["log_return"].abs() > np.log(1.20)
    report["极端值数"] = int(df["is_extreme"].sum())

    df["code"] = code
    df["code"] = df["code"].astype(str).str.zfill(6)
    df["name"] = name
    df = df.sort_index()
    return df, report

# 对所有股票执行清洗
all_cleaned = []
all_reports = []

for stock in stocks:
    fp = f"data/stock/stock_{stock['code']}.csv"
    if os.path.exists(fp):
        df_c, rep = clean_stock(fp, stock["code"], stock["name"])
        all_cleaned.append(df_c)
        all_reports.append(rep)
        print(f"  {stock['name']}({stock['code']}): "
              f"{rep['原始行数']}->{rep['去重后行数']}行, "
              f"重复{rep['重复值数']}, 极端值{rep['极端值数']}")

stock_clean = pd.concat(all_cleaned)
print(f"\n合并后总行数: {len(stock_clean)}")

In [ ]:
summary = pd.DataFrame(all_reports)
display(summary)
print("\n清洗步骤说明：")
print("1. 缺失值检测：统计每列缺失数量和比例")
print("2. 缺失值处理：前向填充（停牌期间价格不变，前值合理）")
print("3. 日期格式：统一为 datetime64 并设为索引")
print("4. 数据类型：确保价格、成交量列为 float64")
print("5. 重复值：删除重复日期行（保留最后一条）")
print("6. 离群值：对日对数收益率超过 ln(1.20) 的标注 is_extreme=True，不删除")
print("\n极端值可能成因：涨跌停板打开、重大利好/利空公告、除权除息等。")

## 3.2 宽表与长表转换

- **宽表**：日期为索引，每列一只股票的收盘价 -> 适合横向对比和可视化
- **长表**：每行一条观测 (date, code, close) -> 适合分组统计和面板回归

In [ ]:
# 收盘价宽表
close_wide = stock_clean.pivot_table(
    index="date", columns="code", values="close", aggfunc="first"
)
close_wide.columns = [f"close_{c}" for c in close_wide.columns]
print("收盘价宽表（前 5 行）：")
display(close_wide.head())
print(f"宽表形状: {close_wide.shape}")

# 宽表转回长表
close_long = close_wide.reset_index().melt(
    id_vars="date", var_name="code", value_name="close"
)
close_long["code"] = close_long["code"].str.replace("close_", "")
print(f"\n长表形状: {close_long.shape}")
display(close_long.head())

## 3.3 多表合并

将个股日度数据与指数、宏观数据合并，每次记录行数变化。

In [ ]:
# 读取沪深 300 指数
hs300 = pd.read_csv("data/index/index_000300.csv", parse_dates=["date"])
hs300["idx_log_return"] = np.log(hs300["idx_close"] / hs300["idx_close"].shift(1))
print(f"沪深 300 行数: {len(hs300)}")

stock_daily = stock_clean[
    ["code", "name", "open", "close", "high", "low",
     "volume", "turnover", "log_return", "is_extreme"]
].copy().reset_index()
print(f"个股数据行数: {len(stock_daily)}")

# Left join 指数
merged = pd.merge(
    stock_daily, hs300[["date", "idx_close", "idx_log_return"]],
    on="date", how="left"
)
print(f"合并指数后行数: {len(merged)} (left join 不增加行，应为 {len(stock_daily)})")

In [ ]:
# 读取 CPI 月度数据
cpi = pd.read_csv("data/macro/macro_cpi.csv")
print(f"CPI 原始列名: {cpi.columns.tolist()}")
display(cpi.head(3))

# 自动识别日期列和值列
date_col = None
val_col = None
for c in cpi.columns:
    if "月" in str(c) or "date" in str(c).lower():
        date_col = c
    if "今值" in str(c) or "当月" in str(c):
        val_col = c
if date_col is None:
    date_col = cpi.columns[0]
if val_col is None:
    for c in cpi.columns[1:]:
        if c != date_col:
            val_col = c
            break

print(f"使用日期列: {date_col}, 值列: {val_col}")

cpi["date"] = pd.to_datetime(cpi[date_col], errors="coerce")
cpi["cpi_yoy"] = pd.to_numeric(cpi[val_col], errors="coerce")
cpi_monthly = cpi[["date", "cpi_yoy"]].dropna().copy()
cpi_monthly["year_month"] = cpi_monthly["date"].dt.to_period("M")
print(f"\nCPI 月度数据（清洗后）：")
display(cpi_monthly.head(10))

In [ ]:
# 将月度 CPI 映射到日度数据
merged["year_month"] = merged["date"].dt.to_period("M")
merged = pd.merge(
    merged, cpi_monthly[["year_month", "cpi_yoy"]],
    on="year_month", how="left"
)
merged = merged.drop(columns=["year_month"])
print(f"合并 CPI 后行数: {len(merged)}, CPI 缺失值: {merged['cpi_yoy'].isnull().sum()}")

# 添加行业信息
merged = pd.merge(merged, stocks_df[["code", "name", "industry"]], on=["code", "name"], how="left")
print(f"合并行业后行数: {len(merged)}")
print(f"\n最终合并数据概览：")
display(merged.head())

## 3.4 数据存储

### 方式 A：CSV（必做）

CSV 格式优点：通用性强、纯文本可读、几乎所有软件支持。
不足：文件体积大、读取慢、不保留数据类型、不支持列式读取。
大规模数据场景下，CSV 的全量加载和字符串解析成为瓶颈。

In [ ]:
os.makedirs("data/clean", exist_ok=True)
os.makedirs("data/combined", exist_ok=True)

stock_clean.to_csv("data/clean/stock_clean.csv", encoding="utf-8-sig")
merged.to_csv("data/combined/combined_data.csv", index=False, encoding="utf-8-sig")

print("CSV 保存完成：")
print(f"  stock_clean.csv: {os.path.getsize('data/clean/stock_clean.csv')/1024:.1f} KB")
print(f"  combined_data.csv: {os.path.getsize('data/combined/combined_data.csv')/1024:.1f} KB")

### 方式 B：Parquet（进阶）

选择 Parquet 的理由：
1. 列式存储，支持只加载需要的列，查询效率高
2. 内置压缩，文件体积比 CSV 小 50%-80%
3. 保留数据类型 Schema，无需重复推断
4. 适合大数据场景和频繁读取操作

In [ ]:
import pyarrow.parquet as pq
import time

# 保存 Parquet
stock_clean.to_parquet("data/clean/stock_clean.parquet")
merged.to_parquet("data/combined/combined_data.parquet")
print("Parquet 保存完成！")

# 演示列式读取
df_small = pd.read_parquet(
    "data/clean/stock_clean.parquet",
    columns=["close", "log_return", "code"]
)
print("\n列式读取（只加载 3 列）：")
display(df_small.head())

# 查看 Schema
schema = pq.read_schema("data/clean/stock_clean.parquet")
print(f"\nParquet Schema:\n{schema}")

# CSV vs Parquet 对比
t0 = time.time()
pd.read_csv("data/clean/stock_clean.csv")
csv_t = time.time() - t0

t0 = time.time()
pd.read_parquet("data/clean/stock_clean.parquet")
pq_t = time.time() - t0

csv_s = os.path.getsize("data/clean/stock_clean.csv") / 1024
pq_s = os.path.getsize("data/clean/stock_clean.parquet") / 1024

print(f"\nCSV     读取耗时: {csv_t:.3f}s  文件大小: {csv_s:.1f} KB")
print(f"Parquet 读取耗时: {pq_t:.3f}s  文件大小: {pq_s:.1f} KB")
print(f"体积压缩比: {pq_s/csv_s*100:.1f}%")

**对比分析**：在本数据规模（约 3 万行 x 10 只股票）下，Parquet 的速度和体积优势有限。
但当数据扩展到全部 A 股（5000+ 只）、季度频率、或多个数据库合并时，
Parquet 的列式读取和压缩优势将显著体现。

In [ ]:
print(f"数据清洗完成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n请继续运行 03_analysis.ipynb 进行描述统计和回归分析")